# RatInABox — Colab Setup

Run each cell in order. By the end you'll have both repos installed, the `.env` configured, and a working simulation ready to launch.

**Before starting:** upload `rat0313_pos.mat` to your Google Drive and note its path.

## 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── CONFIGURE THESE PATHS ────────────────────────────────────────────────────
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/RatInABox'
MATLAB_FILE_PATH  = f'{DRIVE_PROJECT_DIR}/fixtures/rat0313_pos.mat'
# ─────────────────────────────────────────────────────────────────────────────

SAVE_DIRECTORY    = f'{DRIVE_PROJECT_DIR}/saved/results'
TRAINING_DATA_DIR = f'{DRIVE_PROJECT_DIR}/saved/training_data'

os.makedirs(SAVE_DIRECTORY,    exist_ok=True)
os.makedirs(TRAINING_DATA_DIR, exist_ok=True)

print('Drive mounted.')
print(f'MATLAB file : {MATLAB_FILE_PATH}')
print(f'Results dir : {SAVE_DIRECTORY}')
assert os.path.exists(MATLAB_FILE_PATH), (
    f'MATLAB file not found at {MATLAB_FILE_PATH}\n'
    'Upload rat0313_pos.mat to your Drive and update MATLAB_FILE_PATH above.'
)

## 2 — Clone repos

In [ ]:
# ── CONFIGURE THESE URLS ─────────────────────────────────────────────────────
RATINABOX_REPO      = 'https://github.com/Dylan-TerMolen/RatInABox.git'
HANNAHS_CEBRAS_REPO = 'https://github.com/Dylan-TerMolen/Hannahs-CEBRAs.git'
# ─────────────────────────────────────────────────────────────────────────────

!git clone {RATINABOX_REPO}      /content/RatInABox      --depth 1 -q
!git clone {HANNAHS_CEBRAS_REPO} /content/Hannahs-CEBRAs --depth 1 -q
print('Repos cloned.')

## 3 — Install dependencies

In [ ]:
!pip install -q \
    scipy \
    matplotlib \
    pandas \
    scikit-learn \
    seaborn \
    tqdm \
    python-dotenv \
    shapely

# CEBRA — also installs torch
!pip install -q cebra==0.5.0

# Install local packages in editable mode
!pip install -q -e /content/RatInABox
!pip install -q -e /content/Hannahs-CEBRAs

print('Dependencies installed. Restarting runtime...')

# Restart so the newly installed packages are picked up cleanly
import os
os.kill(os.getpid(), 9)


## 4 — Write .env

In [ ]:
env_path = '/content/RatInABox/ratinabox/hsw/.env'

env_contents = (
    '# Auto-generated by colab_setup.ipynb\n'
    f'MATLAB_FILE_PATH={MATLAB_FILE_PATH}\n'
    f'SAVE_DIRECTORY={SAVE_DIRECTORY}\n'
    f'TRAINING_DATA_DIR={TRAINING_DATA_DIR}\n'
    'ENVIRONMENT=colab\n'
)

with open(env_path, 'w') as f:
    f.write(env_contents)

print(f'.env written to {env_path}\n')
print(env_contents)

## 5 — Verify setup

In [ ]:
import sys
sys.path.insert(0, '/content/RatInABox/ratinabox/hsw')

import scipy.io
import ratinabox
import hannahs_cebras
from ratinabox.hsw.env import config

data = scipy.io.loadmat(config.get_matlab_file_path())
print('MATLAB keys:', [k for k in data.keys() if not k.startswith('_')])

print('\n✓ ratinabox version:', ratinabox.__version__)
print('✓ hannahs_cebras imported')
print('✓ config loaded')
print('✓ MATLAB data loaded')
print('\nSetup complete — ready to run simulations.')

## 6 — Run a simulation

Adjust the config block at the top of the cell. Results are saved to your Drive under `saved/results/`.

In [ ]:
# ── CONFIGURE RUN ────────────────────────────────────────────────────────────
MODEL_TYPE      = 'additive'               # additive | dependent | place_dependent
BALANCE_VALUES  = '0.0,0.25,0.5,0.75,1.0' # ignored for place_dependent
RESPONSIVE      = '0.5'
PCT_PLACE_CELLS = '0.5'
NUM_ITERS       = 5
HOLDOVERS       = '1'
# ─────────────────────────────────────────────────────────────────────────────

import subprocess, sys

cmd = [
    sys.executable, '/content/RatInABox/ratinabox/hsw/main.py',
    '--model_type',          MODEL_TYPE,
    '--num_iters',           str(NUM_ITERS),
    '--responsive_values',   RESPONSIVE,
    '--percent_place_cells', PCT_PLACE_CELLS,
    '--holdovers',           HOLDOVERS,
]

if MODEL_TYPE in ('additive', 'dependent'):
    cmd += [
        '--balance_values', BALANCE_VALUES,
        '--balance_dist',   'fixed',
        '--balance_std',    '0.1',
    ]

print('Running:', ' '.join(cmd), '\n')

process = subprocess.Popen(
    cmd,
    cwd='/content/RatInABox/ratinabox/hsw',
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end='', flush=True)

process.wait()
print('\nReturn code:', process.returncode)
